# 01 — Data acquisition and quality audit
This notebook retrieves official mandi data or loads the portal CSV fallback. It never prints the API key. Synthetic data is used only when `SOURCE = 'demo'`.

In [ ]:
%pip install -q -e .

In [ ]:
from pathlib import Path
import pandas as pd

from agridecision.config import get_data_gov_api_key
from agridecision.data.csv_loader import load_mandi_file
from agridecision.data.data_gov import DataGovClient
from agridecision.data.demo import generate_demo_mandi_data
from agridecision.data.quality import validate_mandi_data
from agridecision.data.schema import standardise_mandi_frame

## Choose exactly one source
Use `csv` while the public API is slow. For Colab, upload the official portal file and set its path. Use `api` only after adding `DATA_GOV_API_KEY` to Colab Secrets with notebook access enabled.

In [ ]:
SOURCE = 'demo'  # change to 'csv' or 'api' for official data
CSV_PATH = Path('data/raw/mandi_prices.csv')

if SOURCE == 'api':
    client = DataGovClient(
        api_key=get_data_gov_api_key(),
        resource_id='9ef84268-d588-465a-a308-a864a43d0070',
        page_size=250,
        request_delay_seconds=2.0,
    )
    raw = client.download_csv(
        CSV_PATH, filters={'commodity': 'Onion'}, progress=lambda n, total: print(n, total)
    )
elif SOURCE == 'csv':
    raw = load_mandi_file(CSV_PATH)
elif SOURCE == 'demo':
    raw = generate_demo_mandi_data()
else:
    raise ValueError('SOURCE must be api, csv, or demo')

print('Rows and columns:', raw.shape)
raw.head()

In [ ]:
standard = standardise_mandi_frame(raw)
clean, quarantine, quality = validate_mandi_data(standard)
quality.to_dict()

In [ ]:
audit = {
    'date_min': clean['arrival_date'].min(),
    'date_max': clean['arrival_date'].max(),
    'states': clean['state'].nunique(),
    'markets': clean['market'].nunique(),
    'commodities': clean['commodity'].value_counts().to_dict(),
    'missing_percent': (clean.isna().mean().mul(100).round(2)).to_dict(),
}
audit

In [ ]:
Path('data/processed').mkdir(parents=True, exist_ok=True)
clean.to_csv('data/processed/mandi_prices.csv', index=False)
quarantine.to_csv('data/processed/quarantined_rows.csv', index=False)
print('Saved validated and quarantined tables separately.')